# Target Guided Ordinal Encoding

---

# 1. What is Target Guided Ordinal Encoding?

**Target Guided Ordinal Encoding** is a feature encoding technique where categorical values are replaced by numbers based on their **relationship with the target variable**.

Instead of assigning numbers randomly or based on category order, we calculate the **average target value (mean target)** for each category and then rank the categories according to that value.

In simple words:

> **Categories are ordered according to how strongly they influence the target variable.**

---

# 2. What Problem Does It Solve?

Machine learning models cannot directly understand categorical variables.

Example:

Dataset:

| City   | House Price |
| ------ | ----------: |
| Mumbai |     80 lakh |
| Delhi  |     70 lakh |
| Patna  |     40 lakh |
| Mumbai |     90 lakh |
| Delhi  |     75 lakh |

The model sees:

```
City = Mumbai, Delhi, Patna
```

It cannot understand that Mumbai is associated with higher house prices.

Target Guided Encoding converts:

```
Mumbai → 2
Delhi  → 1
Patna  → 0
```

based on the average house price.

---

# 3. How Does Target Guided Encoding Work?

Let's understand step-by-step.

Suppose we have:

## Dataset

| City   | Price |
| ------ | ----: |
| Mumbai |    90 |
| Mumbai |    80 |
| Delhi  |    70 |
| Delhi  |    60 |
| Patna  |    30 |
| Patna  |    40 |

Target variable:

```
Price
```

---

## Step 1: Calculate Mean Target for Each Category

### Mumbai

[
\frac{90+80}{2}=85
]

### Delhi

[
\frac{70+60}{2}=65
]

### Patna

[
\frac{30+40}{2}=35
]

Result:

| City   | Average Price |
| ------ | ------------: |
| Mumbai |            85 |
| Delhi  |            65 |
| Patna  |            35 |

---

## Step 2: Sort Categories

Highest target mean gets the highest rank.

| City   | Mean Price | Encoding |
| ------ | ---------: | -------: |
| Patna  |         35 |        0 |
| Delhi  |         65 |        1 |
| Mumbai |         85 |        2 |

---

## Step 3: Replace Categories

Original:

| City   |
| ------ |
| Mumbai |
| Delhi  |
| Patna  |

After encoding:

| City   | Encoded |
| ------ | ------: |
| Mumbai |       2 |
| Delhi  |       1 |
| Patna  |       0 |

---

# 4. Mathematical Explanation

For category (c):

[
Encoding(c)=Mean(Target|Category=c)
]

Example:

[
Mumbai = Mean(Price|Mumbai)
]

[
=85
]

Then categories are ranked.

---

# 5. When Should I Use Target Guided Ordinal Encoding?

Use it when:

### 1. Categorical feature has many unique categories

Example:

```
City:
5000 different cities
```

One-Hot Encoding would create:

```
5000 columns
```

Target encoding creates:

```
1 column
```

---

### 2. Category has a relationship with target

Examples:

| Feature          | Target          |
| ---------------- | --------------- |
| City             | House Price     |
| Product Category | Sales           |
| ZIP Code         | Customer Income |
| Brand            | Product Rating  |

---

### 3. Tree-based models

Works well with:

* Decision Trees
* Random Forest
* XGBoost
* LightGBM
* CatBoost

---

# 6. When Should I NOT Use Target Guided Encoding?

## 1. Very small datasets

Because category mean may not be reliable.

Example:

```
Category A:
Only 1 sample
```

Mean will not represent the true behavior.

---

## 2. When there is no relationship between category and target

Example:

```
Favorite Color → Salary
```

Encoding based on salary would introduce noise.

---

## 3. Before train-test split

This is a major mistake.

Wrong:

```
Entire Dataset
      ↓
Target Encoding
      ↓
Train-Test Split
```

The target information from test data leaks into training.

Correct:

```
Split Dataset

      ↓

Calculate encoding only from training data

      ↓

Transform validation/test data
```

---

# 7. Types of Target Encoding

---

# A. Mean Target Encoding

Most common.

Replace category with average target.

Example:

```
City → Average House Price
```

Formula:

[
CategoryValue = \frac{\sum Target}{Number\ of\ observations}
]

---

# B. Smooth Target Encoding

Problem:

Small categories may produce extreme values.

Example:

```
Category A:

1 customer

Target = 100%
```

This is unreliable.

Smoothing combines:

* Category mean
* Overall target mean

Formula:

[
Encoded=
\frac{n \times CategoryMean + m \times GlobalMean}
{n+m}
]

Where:

* n = category sample size
* m = smoothing parameter

---

# C. Leave-One-Out Encoding

When calculating encoding for a row, that row's target is excluded.

Used to reduce overfitting.

---

# D. Probability Ratio Encoding

Uses probability difference:

[
\frac{P(Target=1|Category)}
{P(Target=0|Category)}
]

Common in:

* Classification problems.

---

# 8. Decision Flow

```
Categorical Feature?

          |
          ↓

Does it have natural order?

          |
     +----+----+
     |         |
    Yes        No

     |          |
Ordinal     Number of categories?

Encoding          |
                  |
          +-------+-------+
          |               |
        Few             Many

          |               |

       OHE        Target Guided Encoding
```

---

# 9. Python Implementation

## Example Dataset

```python
import pandas as pd

df = pd.DataFrame({
    'City':['Mumbai','Delhi','Patna','Mumbai','Delhi'],
    'Price':[90,70,40,80,60]
})
```

---

## Step 1: Calculate Mean Target

```python
mean_price = df.groupby('City')['Price'].mean()

print(mean_price)
```

Output:

```
Delhi      65
Mumbai     85
Patna      40
```

---

## Step 2: Map Values

```python
df['City_encoded'] = df['City'].map(mean_price)

print(df)
```

Output:

| City   | Encoded |
| ------ | ------: |
| Mumbai |      85 |
| Delhi  |      65 |
| Patna  |      40 |

---

# Using Category Encoders Library

Install:

```bash
pip install category_encoders
```

Code:

```python
import category_encoders as ce

encoder = ce.TargetEncoder(
    cols=['City']
)

df_encoded = encoder.fit_transform(
    df[['City']],
    df['Price']
)
```

---

# 10. Target Encoding in Classification

Example:

Predict:

```
Customer will churn?
```

Dataset:

| Plan    | Churn |
| ------- | ----: |
| Basic   |     1 |
| Basic   |     1 |
| Premium |     0 |
| Premium |     0 |

Mean:

Basic:

[
\frac{1+1}{2}=1
]

Premium:

[
\frac{0+0}{2}=0
]

Encoding:

| Plan    | Encoded |
| ------- | ------: |
| Basic   |       1 |
| Premium |       0 |

---

# 11. Pros and Cons

## Advantages

✅ Handles high-cardinality features.

Example:

```
10000 cities
```

becomes:

```
1 column
```

---

✅ Captures relationship between feature and target.

---

✅ Works well with tree-based models.

---

## Disadvantages

❌ Risk of target leakage.

❌ Can overfit small categories.

❌ Requires careful validation.

❌ Encoding changes when target distribution changes.

---

# 12. Algorithm Compatibility

| Algorithm           | Works Well? |
| ------------------- | ----------- |
| Linear Regression   | ✅           |
| Logistic Regression | ✅           |
| Decision Tree       | ✅           |
| Random Forest       | ✅           |
| XGBoost             | ✅           |
| LightGBM            | ✅           |
| CatBoost            | ✅           |
| KNN                 | ⚠️ Depends  |
| Neural Networks     | ✅           |

---

# 13. Interview Questions

### Basic

**Q1. What is Target Encoding?**

Answer:

> Target Encoding replaces categories with values calculated from the target variable, usually the mean target value.

---

**Q2. Difference between Ordinal Encoding and Target Encoding?**

| Ordinal               | Target                   |
| --------------------- | ------------------------ |
| Uses predefined order | Uses target relationship |
| No target information | Uses target information  |
| No leakage risk       | Leakage risk             |

---

### Advanced

**Q. Why is target encoding dangerous?**

Because it uses target information, causing data leakage if not performed correctly.

---

**Q. How do you prevent overfitting in target encoding?**

Methods:

* Smoothing
* Cross-validation based encoding
* Leave-one-out encoding

---

# 14. Common Mistakes

❌ Applying target encoding before train-test split.

❌ Using it without checking category frequency.

❌ Ignoring smoothing for rare categories.

❌ Using it when One-Hot Encoding is sufficient.

❌ Allowing the target variable to influence test data preprocessing.

---

# 15. Real-World Examples

## House Price Prediction

Feature:

```
Location
```

Target:

```
House Price
```

Encoding:

```
Mumbai → 85 lakh
Delhi → 65 lakh
Patna → 40 lakh
```

---

## E-commerce

Feature:

```
Product Category
```

Target:

```
Sales Amount
```

Categories are ranked by average sales.

---

## Banking

Feature:

```
Occupation
```

Target:

```
Loan Default
```

Occupations are encoded based on default probability.

---

## Fraud Detection

Feature:

```
Merchant ID
```

Target:

```
Fraud Probability
```

High-risk merchants receive higher encoding values.

---

# 16. Revision Box

```
Target Guided Ordinal Encoding

Purpose:
✔ Convert categorical variables into numbers
✔ Use target relationship

Process:

Category
    ↓
Calculate target mean
    ↓
Rank categories
    ↓
Assign numbers


Best For:
✔ High cardinality features
✔ Categories related to target

Avoid:
✘ Small datasets
✘ Before train-test split
✘ Without leakage prevention


Advantages:
✔ Reduces dimensionality
✔ Captures target relationship

Disadvantages:
✘ Leakage risk
✘ Overfitting risk


Interview Tip:
Target Encoding uses information from the target variable, so it must be performed carefully to avoid data leakage.
```


In [ ]:
# Target guided Encoding technique

In [1]:
import pandas as pd  # Import pandas library; it is used for creating and manipulating data in the form of DataFrames.

# Create a sample DataFrame containing a categorical feature ('city') and a target variable ('price').
df = pd.DataFrame({
    'city': ['New York', 'London', 'Paris', 'Tokyo', 'New York', 'Paris'],
    # 'city' is a categorical column containing different city names.
    # These categories have no natural numerical order, so they need encoding before using in machine learning models.

    'price': [200, 150, 300, 250, 180, 320]
    # 'price' is the target variable.
    # Target guided ordinal encoding uses this target variable to calculate an order/ranking for categories.
    # Here, the average price associated with each city will determine the encoded value.
})

In [6]:
df

# Display the DataFrame.
# It shows the categorical feature ('city') and the target variable ('price') values stored in the DataFrame.
# In machine learning, we use this DataFrame to analyze the relationship between the categorical feature and the target variable.

,city,price
0,New York,200
1,London,150
2,Paris,300
3,Tokyo,250
4,New York,180
5,Paris,320


In [8]:
df.groupby('city')['price'].mean()
# groupby('city') -> Groups the rows of the DataFrame based on unique values in the 'city' column.
# Each city becomes a separate group.

# ['price'] -> Selects only the 'price' column from each group.
# We select the target variable because target guided ordinal encoding uses the target's statistical relationship with categories.

# .mean() -> Calculates the average price for each city group.
# The mean is used to understand the average target value associated with each category.
# These average values will be used to create an ordinal ranking for the categorical variable.

# Output:
# London       150.0
# New York     190.0
# Paris        310.0
# Tokyo        250.0

# Explanation:
# London has the lowest average price, so it gets a lower encoded value.
# Paris has the highest average price, so it gets a higher encoded value.
# The categories are ordered based on their relationship with the target variable ('price').
# This ordering is the main idea behind Target Guided Ordinal Encoding.

,price
city,
London,150.0
New York,190.0
Paris,310.0
Tokyo,250.0


In [10]:
mean_price = df.groupby('city')['price'].mean().to_dict()

In [12]:
df['city_encoded'] = df['city'].map(mean_price)
# Creates a new column named 'city_encoded' in the DataFrame.
# This column will store the encoded numerical values of the 'city' categories.

# df['city'] -> Selects the original categorical column containing city names.
# Example values: ['New York', 'London', 'Paris', 'Tokyo']

# .map(mean_price) -> Replaces each city name with its corresponding mean target value.
# 'mean_price' is a dictionary or Series containing the average price for each city.
# Example:
# mean_price = {
#     'London': 150,
#     'New York': 190,
#     'Tokyo': 250,
#     'Paris': 310
# }

# Mapping process:
# New York  -> 190
# London    -> 150
# Paris     -> 310
# Tokyo     -> 250

# Why we use map():
# map() is used to convert categorical values into numerical values using a predefined mapping.
# In target guided ordinal encoding, the categories are replaced by their relationship with the target variable.

# Result:
# The categorical feature 'city' is converted into a numerical feature 'city_encoded'.
# Machine learning algorithms can now use this encoded feature as input.

In [13]:
df

,city,price,city_encoded
0,New York,200,190.0
1,London,150,150.0
2,Paris,300,310.0
3,Tokyo,250,250.0
4,New York,180,190.0
5,Paris,320,310.0


In [15]:
df[['city','city_encoded']] # this will go now to the model for training purpose

,city,city_encoded
0,New York,190.0
1,London,150.0
2,Paris,310.0
3,Tokyo,250.0
4,New York,190.0
5,Paris,310.0
